# PANDORA Big Five — ML Pipeline Experiment Lab

**Repository:** `Popthemy/personality-prediction-app`  
**Branch:** `pandora`  
**Experiment runner:** `backend/ml_pipeline/experiments/pandora_runner.py`  
**PANDORA ingestion:** `backend/ml_pipeline/services/data/pandora.py`  
**GAN:** `backend/ml_pipeline/services/augmentation/gan.py`

This notebook validates the machine-learning pipeline independently of Django. It uses the Django-free `ExperimentRunner` and does **not** call the Django `PipelineOrchestrator`.


## Experimental pipeline

```text
PANDORA Reddit comments
        ↓
PANDORA ingestion + DataCleaner
        ↓
20-user smoke sample
        ↓
Baseline OR Q-learning
        ↓
BERT embeddings
        ↓
optional GAN augmentation
        ↓
      ┌───────┴───────┐
    Lasso           LSTM
 continuous       3-class
 OCEAN score      prediction
      └───────┬───────┘
              ↓
       metrics + threshold
```

The runner supports the 2 × 2 × 2 design: selection × GAN × model = 8 conditions.

The four research cells are:

- **E1:** Baseline + no GAN
- **E2:** Q-learning + no GAN
- **E3:** Baseline + GAN
- **E4:** Q-learning + GAN

Each cell is evaluated with both Lasso and LSTM.


## Important experimental boundaries

- `N_USERS = 20` is a smoke-test size, not the final experiment.
- The current PANDORA parquet export does not retain literal Reddit usernames; the repository ingestion layer therefore uses its documented proxy-user grouping.
- Q-learning receives cleaned comment text and performs selection before BERT.
- GAN augmentation is applied to the training data only.
- The same participant-level split must be used across comparable conditions.
- BERT embeddings are cached to Drive.
- OCEAN labels are targets for prediction/evaluation, not Q-learning inputs.


In [ ]:
# 1. Google Drive and experiment configuration
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import subprocess
import logging
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/personality_prediction_lab')
DATA_DIR = DRIVE_ROOT / 'datasets'
CACHE_DIR = DRIVE_ROOT / 'bert_cache'
RESULTS_DIR = DRIVE_ROOT / 'results' / '20_user_smoke_test'
REPO_DIR = Path('/content/personality-prediction-app')

for p in (DRIVE_ROOT, DATA_DIR, CACHE_DIR, RESULTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

N_USERS = 20
SEED = 42
RUN_EXPERIMENTS = True

print('Drive:', DRIVE_ROOT)
print('Users:', N_USERS)
print('Results:', RESULTS_DIR)


In [ ]:
# 2. Clone/update the exact pandora branch
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone',
        '--branch', 'pandora',
        '--single-branch',
        'https://github.com/Popthemy/personality-prediction-app.git',
        str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'pandora'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'pandora'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/pandora'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print('Branch:', subprocess.check_output(
    ['git', 'rev-parse', '--abbrev-ref', 'HEAD'], text=True
).strip())
print('Commit:', subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip())


In [ ]:
# 3. Install ML dependencies only
packages = [
    'pandas', 'pyarrow', 'numpy', 'scikit-learn',
    'scipy', 'torch', 'transformers', 'datasets', 'tqdm'
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *packages],
    check=True
)

print('ML dependencies installed.')


In [ ]:
# 4. Import the Django-free experiment engine
from backend.ml_pipeline.services.data.pandora import load_pandora_comments
from backend.ml_pipeline.experiments.pandora_runner import (
    ExperimentConfig,
    ExperimentRunner,
    EXPERIMENTS,
)

print('ExperimentRunner imported.')
print('Configured experiment cells:', list(EXPERIMENTS))


## 5. Download and cache the PANDORA training split

The repository ingestion service expects a local Parquet file. We therefore download the Hugging Face training split once and keep it on Google Drive.


In [ ]:
from datasets import load_dataset
import pandas as pd

PANDORA_PARQUET = DATA_DIR / 'pandora_big5_train.parquet'

if not PANDORA_PARQUET.exists():
    ds = load_dataset('jingjietan/pandora-big5', split='train')
    df = ds.to_pandas()
    df.to_parquet(PANDORA_PARQUET, index=False)
else:
    df = pd.read_parquet(PANDORA_PARQUET)

print('Rows:', len(df))
print('Columns:', list(df.columns))
display(df.head())


In [ ]:
# 6. Prepare PANDORA through the repository ingestion layer
PREPARED_JSON = DATA_DIR / 'pandora_prepared_train.json'

prepared = load_pandora_comments(
    pandora_file=PANDORA_PARQUET,
    output_path=PREPARED_JSON,
    min_text_length=3,
    group_by='traits',
)

print('Prepared proxy-users:', len(prepared))
print('Cleaned comments:', sum(len(u.comments) for u in prepared))


In [ ]:
# 7. Configure the 20-user smoke experiment
cfg = ExperimentConfig(
    sample_n_users=N_USERS,
    min_comments_per_user=5,
    seed=SEED,
    top_k=10,
    qlearning_train_epochs=3,
    val_ratio=0.2,
    synthetic_weight=0.35,
    gan_epochs=150,
    gan_batch_size=16,
    bert_max_length=256,
    lstm_epochs=35,
    lstm_batch_size=4,
    output_dir=str(RESULTS_DIR),
    embedding_cache_dir=str(CACHE_DIR),
)

print(cfg)


## 8. Run E1–E4 through `ExperimentRunner`

This is the main execution cell.

It does not invoke Django. The experiment runner coordinates the existing ML services directly and produces the matched Lasso/LSTM results for the four selection/augmentation cells.


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)

runner = ExperimentRunner(prepared, cfg)
bundle = runner.run() if RUN_EXPERIMENTS else None

if bundle is not None:
    print('Experiment run completed.')
    print('Sample information:', bundle.get('sample'))


In [ ]:
# 9. Main comparison
if bundle is not None:
    display(bundle['comparison'])


In [ ]:
# 10. Lasso versus LSTM under matched experimental conditions
if bundle is not None:
    display(bundle['model_comparison'])


In [ ]:
# 11. Isolated Q-learning and GAN effects
if bundle is not None:
    print('Q-learning effect')
    display(bundle['factor_effects']['qlearning_effect'])

    print('GAN effect')
    display(bundle['factor_effects']['gan_effect'])


In [ ]:
# 12. Threshold / hybrid evaluation results
if bundle is not None:
    for cell, evaluation in bundle['hybrid_cell_evaluations'].items():
        print('\n===', cell, '===')
        print(json.dumps(evaluation, indent=2, default=str)[:12000])


In [ ]:
# 13. Findings summary
if bundle is not None:
    print(json.dumps(bundle['findings'], indent=2, default=str))


In [ ]:
# 14. Verify Drive persistence
for p in sorted(RESULTS_DIR.rglob('*')):
    if p.is_file():
        print(
            p.relative_to(DRIVE_ROOT),
            f'({p.stat().st_size / 1024:.1f} KB)'
        )


## 15. Smoke-test gate before scaling

Before increasing the dataset size, verify:

1. E1–E4 all complete without pipeline errors.
2. The same users/split are used for comparable conditions.
3. Q-learning actually changes the selected comments relative to baseline.
4. GAN augmentation is restricted to training data.
5. BERT embeddings are produced and cached correctly.
6. Lasso produces continuous OCEAN predictions and regression metrics.
7. LSTM produces three-class predictions and classification metrics.
8. Threshold analysis is derived from the continuous Lasso output.
9. Results and caches survive a Colab runtime reset.

Only after these checks pass should the training configuration and user count be increased.
